# Interactivity

In [ ]:
import pandas as pd
import altair as alt

In [ ]:
#gapminder = pd.read_csv("https://raw.githubusercontent.com/OHI-Science/data-science-training/master/data/gapminder.csv")
sdf = pd.read_csv("StudentData26_cleaned.csv")
sdf.loc[sdf['distance'] < 0.1, 'distance'] = 0.1
sdf.loc[sdf['duration'] < 0.1, 'duration'] = 0.1

## Selection
Altair provides three selection options:
* Single Selection
* Interval Selection
* Multi Selection

### Single Selection
The alt.selection_single() function allows the user to click on single chart objects to select them, one at a time.

In [ ]:
# create a selection instance by calling alt.selection_point(), indicating we want a selection defined over a single value
selection = alt.selection_point() # by default, the selection uses a mouse click to determine the selected value

alt.Chart(sdf).mark_circle().encode(
    x='distance:Q',
    y='duration:Q',
    # once our selection has been defined, we can use it as a parameter for conditional encodings, which apply a different encoding depending on whether a data record lies in or out of the selection
    color=alt.condition(selection, 'Creativity:N', alt.value('grey')),
    # an empty selection includes all data points, and so initially all points will be colored
    opacity=alt.condition(selection, alt.value(0.8), alt.value(0.1))
).add_params(
    selection # to register a selection with a chart, we must add it using the .add_params() method
)

Let's compare the three selection types side by side. To keep the code clear, we first define a function (plot) that generates a scatterplot. We can pass the selection to the plot function so that it is applied to the visualization:

In [ ]:
def plot(selection):
    return alt.Chart(sdf).mark_circle().encode(
        x='distance:Q',
        y='duration:Q',
        color=alt.condition(selection, 'Creativity:N', alt.value('grey')),
        opacity=alt.condition(selection, alt.value(0.8), alt.value(0.1))
    ).properties(
        width=240,
        height=180
    ).add_params(
    selection # to register a selection with a chart, we must add it using the .add_params() method
)

The first visualization (single) corresponds to the first example. The second visualization supports shift-click interaction to select multiple points. The third visualization (interval) creates a selection area when dragging with the mouse:

In [ ]:
#since Altair 5 selection_single and selection_multi are depricated - selection_point() does multi
alt.hconcat(
  plot(alt.selection_point(toggle=False, nearest=True)).properties(title='Single, nearest enabled'),
  plot(alt.selection_point()).properties(title='Multi (Shift-Click)'),
  plot(alt.selection_interval()).properties(title='Interval (Drag)'),
)

## Panning & Zooming

The basic interactions that Altair makes available are panning, zooming, and tooltips. This can be done in your chart without any use of the selection interface, using the interactive() shortcut method and the tooltip encoding.

In [ ]:
alt.Chart(sdf).mark_circle().encode(
    x='distance:Q',
    y=alt.Y('duration:Q', axis=alt.Axis(minExtent=30)),
    tooltip=['Creativity:N', 'languages:N', 'distance:Q'],
).properties(
    width=600,
    height=400
).interactive()

By default, zooming and panning includes both axes. However, it is also possible to restrict this interaction to only one axis by using *encodings=['x']*  to restrict the selection to the x-channel.

In [ ]:
alt.Chart(sdf).mark_circle().encode(
    x='distance:Q',
    y=alt.Y('duration:Q', axis=alt.Axis(minExtent=30)),
    tooltip=['Creativity:N', 'languages:N', 'distance:Q'],
).properties(
    width=600,
    height=400
).add_params(
    alt.selection_interval(bind='scales', encodings=['x'])
)

## Filtering

Dynamic queries enable quick exploration of data to isolate interesting patterns. A common approach is to use standard widgets such as sliders, radio buttons and drop-down menus.

The first step is to extract all continents from the Gapminder dataset:

In [ ]:
creativities = ['very good', 'good', 'medium', 'some', 'not really' ]

Now a single selection should be created that is linked to a drop-down menu.

In [ ]:
# 1. Use None for the 'All' logic, and keep the strings for others
creativities_with_all = [None] + creativities
labels = ['All'] + creativities

selectCreativity = alt.selection_point(
    name='Select',
    fields=['Creativity'],
    # 2. Use 'options' for the data values and 'labels' for the dropdown text
    bind=alt.binding_select(options=creativities_with_all, labels=labels, name='Creativity: '),
    value=None  # Sets 'All' as the default starting view
)

condicont = alt.condition(selectCreativity, alt.value(0.75), alt.value(0.05))

alt.Chart(sdf).mark_circle().encode(
    x='distance:Q',
    y='duration:Q',
    opacity=condicont
).add_params(
    selectCreativity
)

### Interactive Legend:
Point your users to what to do! They probably don't know how they can interact with the chart.

In [ ]:
# Use selection_point for multi-select logic
selectTransport = alt.selection_point(
    fields=['transportation'], 
    bind='legend',  # Binds interaction to the legend
    on='click',
    toggle='event.ctrlKey' # Hold Shift to multi-select; change to 'event.ctrlKey' if preferred
)

alt.Chart(sdf).mark_circle().encode(
    x='distance:Q',
    y='duration:Q',
    color='transportation:N', # Ensure mapped to color for the legend to appear
    opacity=alt.condition(selectTransport, alt.value(0.75), alt.value(0.05))
).add_params(
    selectTransport
)

Interactive Points in Chart:

In [ ]:
selectTransportPoint = alt.selection_point(
    fields=['transportation'],
    toggle='true' # Clicking a point adds/removes that continent from the current selection
)

alt.Chart(sdf).mark_circle().encode(
    x='distance:Q',
    y='duration:Q',
    color='transportation:N', # Ensure continent is mapped to color for the legend to appear
    opacity=alt.condition(selectTransportPoint, alt.value(0.75), alt.value(0.05))
).add_params(
    selectTransportPoint
)

### Kombination von mehreren Filtern:

In [ ]:
transport = sdf['transportation'].unique() # get unique field values
transport = list(filter(lambda d: d is not None, transport)) # filter out None values
transport.sort() # sort alphabetically
transport

In [ ]:
# single-value selection over [continent, years] pairs
selection = alt.selection_point(
    name='Select',
    fields=['Creativity', 'transportation'],
    bind={'Creativity': alt.binding_radio(options=creativities_with_all), 'transportation': alt.binding_select(options=transport)}
)

# scatter plot, modify opacity based on selection
alt.Chart(sdf).mark_circle().encode(
    x='distance:Q',
    y='duration:Q',
    color="transportation:N",
    opacity=alt.condition(selection, alt.value(0.75), alt.value(0.05))
).add_params(
    selection
)

### Scented Widgets:

In [ ]:
brush = alt.selection_interval(
    encodings=['x'] # limit selection to x-axis (year) values
)

# dynamic query histogram
years = alt.Chart(sdf).mark_bar().encode(
    alt.X('age:Q'),
    alt.Y('count():Q', title=None)
).properties(
    width=650,
    height=50
).add_params(
    brush
)

# scatter plot, modify opacity based on selection
scatter = alt.Chart(sdf).mark_circle().encode(
    x='distance:Q',
    y='duration:Q',
    opacity=alt.condition(brush, alt.value(0.75), alt.value(0.05))
).properties(
    width=650,
    height=400
)

alt.vconcat(years, scatter).properties(spacing=5)

## Brushing & Linking

In [ ]:
# defines an interval selection
brush = alt.selection_interval(
    resolve='global' # resolve all selections to a single global instance
)

# scatter plot, modify opacity based on selection
scatter1 = alt.Chart(sdf).mark_circle().encode(
    x='distance:Q',
    y='duration:Q',
    opacity=alt.condition(brush, alt.value(0.75), alt.value(0.05))
).properties(
    width=650,
    height=400
).add_params(
    brush
)

# scatter plot, modify opacity based on selection
scatter2 = alt.Chart(sdf).mark_circle().encode(
    x='age:Q',
    y='transportation:N',
    opacity=alt.condition(brush, alt.value(0.75), alt.value(0.05))
).properties(
    width=650,
    height=400
).add_params(
    brush
)


alt.vconcat(scatter1, scatter2).properties(spacing=5)

### Combine with legend-multi-select

In [ ]:
# defines an interval selection
brush = alt.selection_interval(
    resolve='global' # resolve all selections to a single global instance
)


scatter1 = alt.Chart(sdf).mark_circle().encode(
    # Use scale_type directly in the Scale definition
    x=alt.X('distance:Q', scale=alt.Scale(type='log')),
    y=alt.Y('duration:Q', scale=alt.Scale(type='log')),
    color='transportation:N',
    opacity=alt.condition(selectTransport & brush, alt.value(0.75), alt.value(0.05))
).properties(
    width=650, height=400
).add_params(brush, selectTransport) # Altair 6 uses .add_params()

scatter2 = alt.Chart(sdf).mark_circle().encode(
    x=alt.X('distance:Q'),
    y=alt.Y('duration:Q'),
    color='transportation:N',
    opacity=alt.condition(selectTransport & brush, alt.value(0.75), alt.value(0.05))
).properties(
    width=650, height=400
).add_params(brush, selectTransport)

chart = alt.vconcat(scatter1, scatter2).properties(spacing=5)

## Export Chart to HTML

Once you have visualized your data, perhaps you would like to publish it somewhere on the web. This can be done straightforwardly using the [vega-embed JavaScript package](https://github.com/vega/vega-embed). A simple example of a stand-alone HTML document can be generated for any chart using the `Chart.save` method:

In [ ]:
chart.save('chart.html')

**Credits:** *Parts of this notebook are based on UW Interactive Data Lab Visualization Curriculum from Jeffrey Heer.*